# Auditoría de resolución de `implied_volatility` de SPY

## Objetivo

Comprobar si los valores idénticos de IV que aparecen en años diferentes son coincidencias de ejemplos o una cuantización del campo original del proveedor. El alcance se limita a la sonda descriptiva de contratos con 21–45 DTE y strikes a ±2% del cierre de SPY. No se construye IV30 ni se evalúa capacidad predictiva.

## Procedencia y controles

Las cadenas originales se leen desde el proyecto HMM compartido, sin copiarlas ni modificarlas. El script reutilizable deduplica solo por `(fecha de cadena, contractID)`, conserva `implied_volatility` como texto crudo y usa el campo `close` local de SPY exclusivamente para definir la sonda ATM. La fecha/hora exacta de cotización sigue sin verificarse.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root / 'scripts'))
from audit_implied_volatility_resolution import audit_iv_resolution, iv_reconstruction_probe

print(f'Proyecto: {project_root}')
print('Lógica reutilizable: scripts/audit_implied_volatility_resolution.py')

Proyecto: C:\Users\avazq\OneDrive\Escritorio\cienciadatos\portafolio\quant-research-notes\opciones_notas\straddle_ATM_SPY_volatilidad_futura
Lógica reutilizable: scripts/audit_implied_volatility_resolution.py


In [2]:
# Lee las cadenas originales en modo lectura; tarda unos minutos.
audit = audit_iv_resolution()
print(f"Respuestas/sesiones recorridas: {audit['files_scanned']:,} / {audit['sessions_scanned']:,}")
print(f"Contratos candidatos únicos: {audit['candidate_unique_session_contracts']:,}")
print(f"Duplicados excluidos del conteo: {audit['duplicate_candidate_records']:,}")
print('Regla fija:', audit['probe'])

Respuestas/sesiones recorridas: 1,674 / 1,674
Contratos candidatos únicos: 343,228
Duplicados excluidos del conteo: 192
Regla fija: {'dte_min': 21, 'dte_max': 45, 'abs_moneyness_max': 0.02, 'spot_field': 'close', 'price_path': 'C:\\Users\\avazq\\OneDrive\\Escritorio\\cienciadatos\\portafolio\\quant-research-notes\\HMM_regime_options\\data\\derived\\spy_daily_adjusted_2020_2026.csv'}


In [3]:
year_table = audit['by_year'].copy()
year_table['top_five'] = year_table['top_five'].map(
    lambda rows: '; '.join(f"{row['iv_raw']} ({row['contracts']:,})" for row in rows)
)
year_table['top_five_pct'] = year_table['top_five_pct'].map(lambda value: f'{value:.2f}%')
display(year_table.rename(columns={
    'year': 'año', 'unique_session_contracts': 'contratos únicos',
    'nonmissing_iv': 'IV no ausente', 'distinct_iv_raw': 'IV distintas',
    'top_five': 'cinco IV más frecuentes', 'top_five_pct': 'concentración top 5'
})[['año', 'contratos únicos', 'IV no ausente', 'IV distintas', 'cinco IV más frecuentes', 'concentración top 5']])

top_ten = pd.DataFrame(audit['top_ten'])
display(Markdown(f"**Precisión original:** {audit['precision_decimals']}.  **IV distintas globales:** {audit['distinct_iv_raw']}.  **Concentración top 10:** {audit['top_ten_pct']:.2f}%."))
display(top_ten.rename(columns={'iv_raw': 'IV literal original', 'contracts': 'contratos'}))

,año,contratos únicos,IV no ausente,IV distintas,cinco IV más frecuentes,concentración top 5
0,2020,52092,52092,87,"0.19048 (3,384); 0.18073 (3,108); 0.20024 (3,0...",29.20%
1,2021,65224,65224,29,"0.14170 (8,302); 0.13195 (8,231); 0.12219 (7,1...",56.07%
2,2022,57582,57582,30,"0.24902 (5,185); 0.23926 (4,789); 0.21000 (4,3...",39.81%
3,2023,33966,33966,24,"0.13195 (4,048); 0.12219 (3,580); 0.14170 (3,3...",50.27%
4,2024,46440,46440,32,"0.12219 (8,619); 0.11244 (8,399); 0.13195 (6,9...",71.41%
5,2025,49102,49102,45,"0.13195 (7,411); 0.14170 (7,384); 0.15146 (6,3...",63.27%
6,2026,38822,38822,22,"0.14170 (6,681); 0.13195 (5,845); 0.15146 (5,7...",67.42%


**Precisión original:** {5: 343228}.  **IV distintas globales:** 89.  **Concentración top 10:** 69.69%.

,IV literal original,contratos
0,0.13195,34346
1,0.14170,32436
2,0.12219,29976
3,0.15146,28251
4,0.16122,24161
5,0.11244,22954
6,0.17097,20584
7,0.18073,17549
8,0.19048,15489
9,0.20024,13439


## Sonda acotada: reconstrucción de IV desde el punto medio

Las fechas quedaron fijadas antes de cualquier inversión: 2020-01-02, 2022-10-03 y 2025-04-03. Para cada fecha se elegirá el vencimiento con DTE más cercano a 30 (empate: vencimiento anterior) y, dentro de él, el strike más cercano al cierre **sin ajustar** de SPY para un call y un put (empate: strike menor, luego `contractID`). Se muestra el punto medio solo si bid > 0 y ask ≥ bid.

### Modelo que requeriría una inversión válida

SPY tiene opciones americanas. La inversión apropiada sería un árbol binomial/CRR o diferencias finitas de ejercicio americano con dividendos discretos: en cada fecha ex-dividendo conocida en t se reduce el subyacente por el importe de efectivo y se permite ejercicio temprano. Para cada contrato se buscaría numéricamente la volatilidad que iguala el precio del modelo al punto medio. Una tasa de dividendo continua sería solo una aproximación que debería declararse y someterse a sensibilidad; no se usará aquí como sustituto.

Los insumos mínimos son: spot sin ajustar, strike, vencimiento, tipo, una curva cero libre de riesgo fechada en t y el calendario con importes de dividendos futuros conocidos en t hasta el vencimiento.

In [4]:
reconstruction_probe = iv_reconstruction_probe()
display(reconstruction_probe[[
    'observation_date', 'spot_close_unadjusted', 'contractID', 'type', 'expiration', 'dte',
    'strike', 'bid', 'ask', 'midpoint', 'provider_iv_raw', 'delta',
    'same_day_recorded_dividend', 'risk_free_curve_available_at_t',
    'forward_dividend_schedule_available_at_t', 'inversion_status'
]])

usable_quotes = (reconstruction_probe['bid'] > 0) & (reconstruction_probe['ask'] >= reconstruction_probe['bid'])
print(f'Cotizaciones utilizables para el punto medio: {usable_quotes.sum()} de {len(reconstruction_probe)}')
print('Tasa fechada en t disponible:', reconstruction_probe['risk_free_curve_available_at_t'].any())
print('Calendario de dividendos futuros conocido en t disponible:', reconstruction_probe['forward_dividend_schedule_available_at_t'].any())

,observation_date,spot_close_unadjusted,contractID,type,expiration,dte,strike,bid,ask,midpoint,provider_iv_raw,delta,same_day_recorded_dividend,risk_free_curve_available_at_t,forward_dividend_schedule_available_at_t,inversion_status
0,2020-01-02,324.87,SPY200131C00325000,call,2020-01-31,29,325.0,3.88,3.91,3.895,0.10268,0.51723,0.0,False,False,BLOCKED: missing dated risk-free curve and for...
1,2020-01-02,324.87,SPY200131P00325000,put,2020-01-31,29,325.0,3.51,3.54,3.525,0.10268,-0.48277,0.0,False,False,BLOCKED: missing dated risk-free curve and for...
2,2022-10-03,366.61,SPY221102C00367000,call,2022-11-02,30,367.0,11.82,11.87,11.845,0.27829,0.52324,0.0,False,False,BLOCKED: missing dated risk-free curve and for...
3,2022-10-03,366.61,SPY221102P00367000,put,2022-11-02,30,367.0,10.85,10.90,10.875,0.26853,-0.47705,0.0,False,False,BLOCKED: missing dated risk-free curve and for...
4,2025-04-03,536.70,SPY250502C00537000,call,2025-05-02,29,537.0,16.88,16.95,16.915,0.26853,0.53026,0.0,False,False,BLOCKED: missing dated risk-free curve and for...
5,2025-04-03,536.70,SPY250502P00537000,put,2025-05-02,29,537.0,14.46,14.57,14.515,0.24902,-0.46965,0.0,False,False,BLOCKED: missing dated risk-free curve and for...


Cotizaciones utilizables para el punto medio: 6 de 6
Tasa fechada en t disponible: False
Calendario de dividendos futuros conocido en t disponible: False


### Conclusión limitada de la sonda

Las seis cotizaciones seleccionadas tienen bid/ask utilizables y muestran el IV del proveedor, pero la inversión queda **BLOQUEADA**. El repositorio contiene `dividend_amount` realizado en la fecha de observación (0.0 en estas tres fechas), no un calendario ni importes de dividendos futuros conocidos entonces; tampoco contiene una curva de tasas libre de riesgo fechada. Por ello no se calcularon IV reconstruidas, comparación ni sensibilidad: hacerlo exigiría inventar tasas o dividendos y podría introducir información posterior.

La factibilidad de reconstruir IV con los datos locales es, por ahora, **INCIERTA**. Para desbloquear esta prueba harían falta, para cada fecha, una curva de tasas o tasa cero con vencimiento coherente y un calendario de ex-dividendos/efectivo disponible en t para el intervalo hasta la expiración. Esta conclusión no construye una serie diaria ni evalúa volatilidad futura.

### Interpretación

Cinco decimales en el texto no implican cinco decimales de resolución efectiva. La tabla permite ver la concentración de una escalera pequeña de valores por año, antes de decidir cómo —o si— usar la IV para una serie diaria.

In [5]:
# Evidencia trazable dentro de respuestas completas: misma IV, strikes y cotizaciones distintos.
for chain_date, example in audit['examples'].items():
    display(Markdown(
        f"### {chain_date}: IV literal `{example['iv_raw']}`  \n"
        f"Archivo: `{example['source_file']}`  \n"
        f"Cierre SPY: {example['spot_close']:.2f}; contratos con esa IV: {example['same_iv_contracts']}; "
        f"strikes distintos: {example['distinct_strikes']}; bid/ask distintos: {example['distinct_bid_ask']}."
    ))
    display(pd.DataFrame(example['contracts']))

### 2020-01-02: IV literal `0.11244`  
Archivo: `C:\Users\avazq\OneDrive\Escritorio\cienciadatos\portafolio\quant-research-notes\HMM_regime_options\data\raw\spy_historical_options_2020-01-02.json.gz`  
Cierre SPY: 324.87; contratos con esa IV: 54; strikes distintos: 13; bid/ask distintos: 54.

,contractID,type,expiration,strike,implied_volatility,bid,ask,last,delta
0,SPY200124C00320500,call,2020-01-24,320.50,0.11244,6.44,6.48,6.37,0.70480
1,SPY200124C00321000,call,2020-01-24,321.00,0.11244,6.04,6.08,5.80,0.68502
2,SPY200124C00321500,call,2020-01-24,321.50,0.11244,5.64,5.68,5.54,0.66473
3,SPY200124C00322000,call,2020-01-24,322.00,0.11244,5.25,5.29,5.18,0.64398
4,SPY200124P00319500,put,2020-01-24,319.50,0.11244,1.52,1.54,1.94,-0.25737
5,SPY200124P00320000,put,2020-01-24,320.00,0.11244,1.60,1.63,1.65,-0.27598
6,SPY200124P00320500,put,2020-01-24,320.50,0.11244,1.69,1.72,2.01,-0.29520
7,SPY200124P00321000,put,2020-01-24,321.00,0.11244,1.79,1.81,1.88,-0.31498


### 2022-10-03: IV literal `0.26853`  
Archivo: `C:\Users\avazq\OneDrive\Escritorio\cienciadatos\portafolio\quant-research-notes\HMM_regime_options\data\raw\spy_historical_options_2022-10-03.json.gz`  
Cierre SPY: 366.61; contratos con esa IV: 63; strikes distintos: 14; bid/ask distintos: 62.

,contractID,type,expiration,strike,implied_volatility,bid,ask,last,delta
0,SPY221024C00368000,call,2022-10-24,368.00,0.26853,9.19,9.24,8.70,0.50038
1,SPY221024C00370000,call,2022-10-24,370.00,0.26853,8.12,8.17,8.12,0.46685
2,SPY221024C00371000,call,2022-10-24,371.00,0.26853,7.63,7.67,7.67,0.45023
3,SPY221024C00372000,call,2022-10-24,372.00,0.26853,7.14,7.19,6.74,0.43373
4,SPY221024P00362000,put,2022-10-24,362.00,0.26853,7.06,7.11,7.53,-0.39891
5,SPY221024P00364000,put,2022-10-24,364.00,0.26853,7.81,7.85,8.16,-0.43225
6,SPY221024P00365000,put,2022-10-24,365.00,0.26853,8.19,8.25,8.50,-0.44906
7,SPY221024P00366000,put,2022-10-24,366.00,0.26853,8.61,8.66,9.12,-0.46590


### 2025-04-03: IV literal `0.25877`  
Archivo: `C:\Users\avazq\OneDrive\Escritorio\cienciadatos\portafolio\quant-research-notes\HMM_regime_options\data\raw\spy_historical_options_2025-04-03.json.gz`  
Cierre SPY: 536.70; contratos con esa IV: 44; strikes distintos: 22; bid/ask distintos: 43.

,contractID,type,expiration,strike,implied_volatility,bid,ask,last,delta
0,SPY250425C00543000,call,2025-04-25,543.00,0.25877,11.55,11.60,11.56,0.45587
1,SPY250425C00544000,call,2025-04-25,544.00,0.25877,11.02,11.07,11.01,0.44441
2,SPY250425C00545000,call,2025-04-25,545.00,0.25877,10.50,10.57,10.46,0.43301
3,SPY250425C00546000,call,2025-04-25,546.00,0.25877,9.99,10.05,9.90,0.42169
4,SPY250425C00547000,call,2025-04-25,547.00,0.25877,9.50,9.56,9.42,0.41046
5,SPY250425P00535000,put,2025-04-25,535.00,0.25877,12.22,12.32,12.11,-0.45114
6,SPY250425P00536000,put,2025-04-25,536.00,0.25877,12.65,12.70,12.55,-0.46280
7,SPY250425P00537000,put,2025-04-25,537.00,0.25877,13.00,13.10,12.91,-0.47447


## Hallazgo y límite

El mismo IV literal aparece en contratos con diferentes strikes, tipos y cotizaciones dentro de la misma respuesta. Esto demuestra que las coincidencias no son solo una consecuencia de haber mostrado ejemplos ATM de años distintos. Aún no se puede afirmar que la IV sea una medición suficientemente granular y específica por contrato para una serie IV30 informativa; la conclusión de esta auditoría es **INCIERTO**. Haría falta documentación verificable de la cuantización/método del proveedor o una fuente original contrastable.

La disponibilidad antes de la apertura de t+1 tampoco puede probarse con estas cadenas: no contienen una hora de cotización verificable.

## Mis notas

- 
- 
- 